# 스트리밍과 배치, 재시도-폴백

- Runnable 에는 `a-` 비동기 메서드 제공

| 기본 | 비동기 | 사용법 |
| --- | --- | --- |
| chain.invoke() | chain.ainvoke(…) | result = await chain.ainvoke() |
| chain.batch() | chain.abatch([…]) | result = await chain.abatch() |
| chain.stream() | chain.astream(…) | async for piece in chain.astream(): |

## 스트리밍 astream()

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Tue" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W4" / "Tue"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [ ]:
import asyncio
import sys

loop = asyncio.get_event_loop()

print("플랫폼 : ", sys.platform)
print("이벤트 루프 : ", type(loop).__name__)
print("돌고 있는지 : ", loop.is_running()) # 주피터 노트북에서는 이벤트 루프가 하나 돌고 있다.

In [ ]:
# 코루틴 (비동기 함수)
async def hello() -> str:
    await asyncio.sleep(0)
    return "안녕하세요"

coro = hello()
try:
    asyncio.run(coro) # 주피터 노트북에서 이 코드 사용 불가
except RuntimeError as e:
    print("async run() : ", f"{type(e).__name__} : {e}")
finally:
    coro.close()

# 노트북에서 사용
value = await hello()
print(value)

In [ ]:
from app.agent.chain import build_answer_chain, load_prompt
from app.integrations.ports import LLMResult

class MockLLM:
    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        return LLMResult(
            text="부산 출장 일비는 1일 5만원입니다.",
            model="mock",
            input_tok=10,
            output_tok=12,
            cost_krw=0.0,
            latency_ms=1,
        )

chain = build_answer_chain(MockLLM(), system_prompt=load_prompt("answer_system"))

# 동기
out = chain.invoke({"question": "부산 출장 일비는?"})
# 비동기
async_out = await chain.ainvoke({"question": "부산 출장 일비는?"})

print("invoke:", out)
print("async_out:", async_out)
# answer가 동기함수라도 비동기 호출시 돌아가긴함 -> 대신 비동기가 아닌 동기 방식으로 돈다 -> 비동기화

In [ ]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([("human", "{question}")])
QUESTION = {"question": "부산 출장 일비는?"}

def fake_stream_llm() -> GenericFakeChatModel:
    return GenericFakeChatModel(messages=iter([AIMessage(content="일비 5만원 - 숙박 실비")]))

pieces = [p async for p in (prompt | fake_stream_llm() | StrOutputParser()).astream(QUESTION)]
print("조각 개수 :", len(pieces))
print("조각 목록 :", pieces)

print("흘려 찍기 : ", end="")
async for piece in (prompt | fake_stream_llm() | StrOutputParser()).astream(QUESTION):
    print(piece, end="", flush=True)
print()

In [ ]:
# buffer 생성
buf = []

# 조각내서 buf 저장
async for piece in (prompt | fake_stream_llm() | StrOutputParser()).astream(QUESTION):
    print(piece, end="", flush=True)
    buf.append(piece)
print()

# 목록의 문자열을 구분자 없이 이어 붙이기. 조각 복원
joined = "".join(buf)
print("합친 결과 : ", joined)

In [ ]:
from langchain_core.runnables import RunnableLambda

count_pure = len(list((prompt | fake_stream_llm() | StrOutputParser()).stream(QUESTION)))

print("llm | parser 조각 : ", count_pure)

decorated = prompt | fake_stream_llm() | StrOutputParser() | RunnableLambda(lambda s: f"[일반] {s}")
count_lambda = len(list(decorated.stream(QUESTION)))

print("llm | parser | runnablelambda 조각 : ", count_lambda)

app_chain = build_answer_chain(MockLLM(), system_prompt=load_prompt("answer_system"))
count_app = len(list(app_chain.stream(QUESTION)))

print("우리 앱 chain.py 조각 :", count_app)

- **TODO** : LLMPort 비동기 지원하도록 넓히기 (추후 진행)

## abatch(), gather()

| 사용방법 | 어떻게 처리? | 언제사용 |
| --- | --- | --- |
| await **chain.abatch**([…]) | LangChain이 동시 실행을 대신 해준다. | **같은 체인에 여러 입력** |
| await **asyncio.gather**(*[…]) | 코루틴 여러개를 모아서 | **서로 다른 일을 동시에** |

In [ ]:
import time

async def fetch_meta(payload: dict) -> str:
    await asyncio.sleep(0.3)
    return f"{payload['doc_id']} 조회 완료"

slow_chain = RunnableLambda(fetch_meta)
inputs = [{"doc_id": "DOC-HR-014"}, {"doc_id": "DOC-PU-007"}, {"doc_id": "DOC-SE-003"}]

# 1.
started = time.perf_counter()
seq_out = []
for one in inputs:
    seq_out.append(await slow_chain.ainvoke(one))
seq_sec = time.perf_counter() - started

print(f"순차 (for + await) : {seq_sec:.2f}초")

# 2.
started = time.perf_counter()
gather_out = list(await asyncio.gather(*[slow_chain.ainvoke(one) for one in inputs]))
gather_sec = time.perf_counter() - started

print(f"asyncio.gather     : {gather_sec:.2f}초")

# 3.
started = time.perf_counter()
batch_out = await slow_chain.abatch(inputs)
batch_sec = time.perf_counter() - started

print(f"abatch             : {batch_sec:.2f}초")

In [ ]:
class SlowMockLLM:
    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        time.sleep(0.3)
        return LLMResult(text="부산 출장 일비는 1일 2만원입니다.", model="mock", latency_ms=300)

async def fetch_doc_meta(payload: dict) -> str:
    await asyncio.sleep(0.3)
    return f"{payload['doc_id']} 국내출장 여비 규정 v2.0 · 인사총무 · 일반"

answer_chain = build_answer_chain(SlowMockLLM(), system_prompt=load_prompt("answer_system"))
meta_chain = RunnableLambda(fetch_doc_meta)

#1. ainvoke
started = time.perf_counter()
answer_seq = await answer_chain.ainvoke({"question": "부산 출장 일비는?"})
meta_seq = await meta_chain.ainvoke({"doc_id": "DOC-HR-014"})
seq_sec = time.perf_counter() - started

#2. gather
started = time.perf_counter()
answer, meta = await asyncio.gather(
    answer_chain.ainvoke({"question": "부산 출장 일비는?"}),
    meta_chain.ainvoke({"doc_id": "DOC-HR-014"}),
)
gather_sec = time.perf_counter() - started

print("답변 :", answer)
print("메타 :", meta)
print(f"순차 : {seq_sec:.2f}초")
print(f"동시 : {gather_sec:.2f}초")

## 재시도 with_retry()

```python
# chat_service.py
for attempt in range(1, MAX_ATTEMPTS + 1):
    result = llm.answer(...)
    try:
        ...
    except ValidationError:
        ...

# LCEL
safe = chain.with_retry(stop_after_attempt=3)
```

In [ ]:
calls = {"n": 0}

def flaky(payload: str) -> str:
    calls["n"] += 1
    if calls["n"] < 3:
        raise RuntimeError(f"일시 오류 {calls['n']}")
    return f"{payload} 성공"

calls["n"] = 0
ok = RunnableLambda(flaky).with_retry(stop_after_attempt=3, wait_exponential_jitter=False)
print("1. stop_after_attempt=3 :", ok.invoke("부산 출장 일비는?"), f"  (실제 호출 {calls['n']}회)")

calls["n"] = 0
tight = RunnableLambda(flaky).with_retry(stop_after_attempt=2, wait_exponential_jitter=False)
try:
    tight.invoke("부산 출장 일비는?")
except RuntimeError as exc:
    print("2. stop_after_attempt=2 :", f"{type(exc).__name__}: {exc}", f"  (실제 호출 {calls['n']}회)")

## 폴백 with_fallbacks()

```python
safe = chain.with_fallbacks([backup_chain, RunnableLambda(fallback_notice)])
```

In [ ]:
from langchain_core.language_models import FakeListChatModel

# 언제나 실패하는 주 기능
def always_fail(payload: dict) -> str:
    raise RuntimeError("주 경로 실패")

# 폴백 응답
def fallback_notice(payload: dict) -> str:
    print("[경고] 주 체인 실패 · 폴백 응답을 반환합니다")  # 실 코드에서는 로그로 남기기
    return "지금은 답변할 수 없습니다. 총무팀 담당자에게 문의해 주세요."

# 1. invoke 의 폴백
safe = RunnableLambda(always_fail).with_fallbacks([RunnableLambda(fallback_notice)])
print("폴백 결과 :", safe.invoke({"question": "부산 출장 일비는?"}))
print("   타입      :", type(safe).__name__)

# 2. 스트리밍 실패를 네트워크없이 테스트해보는 모델
flaky_model = FakeListChatModel(responses=["부산 출장 일비는 2만원입니다"], error_on_chunk_number=2)
try:
    list(flaky_model.stream("부산 출장 일비는?"))
except Exception as exc:
    print("스트리밍 실패 :", type(exc).__name__)

# 3. 같은 실패에서 fallback 걸기
first_chunk_fail = FakeListChatModel(responses=["부산 출장 일비는 2만원입니다"], error_on_chunk_number=0)
stream_safe = (first_chunk_fail | StrOutputParser()).with_fallbacks([RunnableLambda(fallback_notice)])
print("폴백이 받은 뒤 :", "".join(stream_safe.stream("부산 출장 일비는?")))